# 🦆 Laboratório: Processamento de Dados com DuckDB e Pandas

> **Objetivo:** Aprender a usar o DuckDB como motor de banco de dados embutido no Python para manipular grandes volumes de dados de forma eficiente — mesmo em máquinas com recursos limitados (ex: 8 GB de RAM).

---

## 📋 Pré-requisitos

| Requisito | Versão Mínima |
|-----------|---------------|
| Python    | 3.9+          |
| DuckDB    | 0.10+         |
| Pandas    | 1.5+          |
| NumPy     | 1.23+         |

## 🗺️ Mapa do Laboratório

1. [Configuração do Ambiente](#1.-Configuração-do-Ambiente)
2. [DuckDB como Banco de Dados Persistente](#2.-DuckDB-como-Banco-de-Dados-Persistente)
3. [Integração de Alta Performance com Pandas](#3.-Integração-de-Alta-Performance-com-Pandas)
4. [Processando Arquivos Grandes com Pouca RAM](#4.-Processando-Arquivos-Grandes-com-Pouca-RAM)
5. [Desafio Prático](#5.-Desafio-Prático)
6. [Referência de Comandos](#6.-Referência-de-Comandos)


In [ ]:
# 🎨 Tema escuro — execute esta célula para ativar
# (Não é necessário no VS Code com tema escuro ativo)
from IPython.display import display, HTML

display(HTML("""
<style>
  /* ── Jupyter Lab / Notebook dark theme ── */
  :root {
    --duck-bg:       #0d1117;
    --duck-surface:  #161b22;
    --duck-border:   #30363d;
    --duck-text:     #e6edf3;
    --duck-muted:    #8b949e;
    --duck-blue:     #58a6ff;
    --duck-orange:   #ffa657;
    --duck-code:     #f0883e;
  }

  /* Notebook background */
  #notebook, .jp-Notebook,
  .jp-Cell, .jp-MarkdownCell, .jp-CodeCell {
    background: var(--duck-bg) !important;
  }

  /* Markdown rendered text */
  .jp-RenderedMarkdown, .jp-MarkdownOutput,
  .text_cell_render {
    color: var(--duck-text) !important;
    background: var(--duck-bg) !important;
  }

  /* Headings */
  .jp-RenderedMarkdown h1, .text_cell_render h1 { color: var(--duck-orange) !important; }
  .jp-RenderedMarkdown h2, .text_cell_render h2 {
    color: var(--duck-blue) !important;
    border-bottom: 1px solid var(--duck-border);
    padding-bottom: 4px;
  }
  .jp-RenderedMarkdown h3, .text_cell_render h3 { color: #79c0ff !important; }

  /* Code editor */
  .jp-Editor .cm-editor, .CodeMirror, .cm-s-jupyter {
    background: var(--duck-surface) !important;
    color: var(--duck-text) !important;
  }
  .cm-editor .cm-gutters { background: var(--duck-surface) !important; border-right: 1px solid var(--duck-border) !important; }

  /* Output area */
  .jp-OutputArea-output, .output_area pre,
  .jp-RenderedText pre {
    background: var(--duck-surface) !important;
    color: var(--duck-text) !important;
  }

  /* Tables */
  .jp-RenderedMarkdown table, .text_cell_render table { border-collapse: collapse; width: 100%; }
  .jp-RenderedMarkdown th, .text_cell_render th {
    background: #21262d !important;
    color: var(--duck-blue) !important;
    border: 1px solid var(--duck-border) !important;
    padding: 6px 12px;
  }
  .jp-RenderedMarkdown td, .text_cell_render td {
    border: 1px solid var(--duck-border) !important;
    padding: 6px 12px;
    color: var(--duck-text) !important;
  }
  .jp-RenderedMarkdown tr:nth-child(even) td { background: var(--duck-surface) !important; }
  .jp-RenderedMarkdown tr:nth-child(odd)  td { background: var(--duck-bg) !important; }

  /* Blockquotes */
  .jp-RenderedMarkdown blockquote, .text_cell_render blockquote {
    border-left: 4px solid var(--duck-blue) !important;
    background: var(--duck-surface) !important;
    color: var(--duck-muted) !important;
    padding: 8px 16px;
    margin: 12px 0;
    border-radius: 0 6px 6px 0;
  }

  /* Inline code */
  .jp-RenderedMarkdown code, .text_cell_render code {
    background: #21262d !important;
    color: var(--duck-code) !important;
    border-radius: 4px;
    padding: 2px 5px;
  }

  /* Code blocks in markdown */
  .jp-RenderedMarkdown pre, .text_cell_render pre {
    background: var(--duck-surface) !important;
    border: 1px solid var(--duck-border) !important;
    border-radius: 6px;
    padding: 12px;
    color: var(--duck-text) !important;
  }
  .jp-RenderedMarkdown pre code, .text_cell_render pre code {
    background: transparent !important;
    color: var(--duck-text) !important;
    padding: 0;
  }

  /* Links */
  .jp-RenderedMarkdown a, .text_cell_render a { color: var(--duck-blue) !important; }

  /* Cell input border accent */
  .jp-Cell-inputWrapper { border-left: 3px solid var(--duck-border) !important; }
  .jp-Cell-inputWrapper:hover { border-left: 3px solid var(--duck-blue) !important; }

  /* Pandas DataFrames */
  .dataframe th { background: #21262d !important; color: var(--duck-blue) !important; }
  .dataframe td { background: var(--duck-surface) !important; color: var(--duck-text) !important; }
  .dataframe { border: 1px solid var(--duck-border) !important; }
</style>
"""))


---

<a id="1.-Configuração-do-Ambiente"></a>

## 1. Configuração do Ambiente

Instale as bibliotecas necessárias e verifique as versões. Execute a célula abaixo — se tudo carregar sem erros, o ambiente está pronto.


In [ ]:
# Descomente a linha abaixo caso não tenha as bibliotecas instaladas
# !pip install duckdb pandas numpy

import duckdb
import pandas as pd
import numpy as np

print(f"✅ DuckDB  : {duckdb.__version__}")
print(f"✅ Pandas  : {pd.__version__}")
print(f"✅ NumPy   : {np.__version__}")
print("\nAmbiente configurado com sucesso!")

---

<a id="2.-DuckDB-como-Banco-de-Dados-Persistente"></a>

## 2. DuckDB como Banco de Dados Persistente

### 💡 Conceito

O DuckDB é o **substituto moderno do SQLite para análise de dados**. Diferente de servidores como PostgreSQL ou MySQL, ele roda dentro do seu processo Python e guarda tudo em um único arquivo local — sem instalação de servidor, sem configuração de rede.

<div style='margin:16px 0'>
<svg viewBox='0 0 520 90' xmlns='http://www.w3.org/2000/svg' style='width:100%;max-width:520px;font-family:monospace'>
  <!-- Python App box -->
  <rect x='10' y='15' width='140' height='60' rx='6' fill='#161b22' stroke='#ffa657' stroke-width='1.5'/>
  <text x='80' y='42' text-anchor='middle' fill='#ffa657' font-size='13' font-weight='bold'>Python App</text>
  <text x='80' y='60' text-anchor='middle' fill='#8b949e' font-size='12'>/ Jupyter</text>
  <!-- Arrow left -->
  <line x1='152' y1='45' x2='195' y2='45' stroke='#58a6ff' stroke-width='1.5' marker-end='url(#arr)'/>
  <!-- SQL label -->
  <text x='255' y='22' text-anchor='middle' fill='#58a6ff' font-size='12' font-weight='bold'>SQL</text>
  <!-- Arrow right -->
  <line x1='315' y1='45' x2='358' y2='45' stroke='#58a6ff' stroke-width='1.5' marker-start='url(#arrL)'/>
  <!-- Double arrow middle -->
  <line x1='152' y1='45' x2='358' y2='45' stroke='#58a6ff' stroke-width='1.5'/>
  <polygon points='152,41 162,45 152,49' fill='#58a6ff'/>
  <polygon points='358,41 348,45 358,49' fill='#58a6ff'/>
  <!-- DuckDB box -->
  <rect x='360' y='15' width='150' height='60' rx='6' fill='#161b22' stroke='#ffa657' stroke-width='1.5'/>
  <text x='435' y='42' text-anchor='middle' fill='#ffa657' font-size='13' font-weight='bold'>DuckDB (embutido)</text>
  <text x='435' y='60' text-anchor='middle' fill='#8b949e' font-size='12'>meubanco.db 💾</text>
</svg>
</div>

> **Quando usar banco persistente?** Quando você quer reutilizar os dados entre sessões do Jupyter sem reprocessar tudo.


In [ ]:
# Criar ou conectar a um banco de dados no disco
# Se o arquivo não existir, o DuckDB cria automaticamente
con = duckdb.connect("meubanco.db")

print("✅ Conectado ao banco 'meubanco.db'")

In [ ]:
# Criar a tabela (IF NOT EXISTS evita erros ao rodar novamente)
con.sql("""
    CREATE TABLE IF NOT EXISTS usuarios (
        id   INTEGER PRIMARY KEY,
        nome TEXT NOT NULL
    )
""")

# Limpar dados antigos para evitar duplicatas ao reexecutar
con.sql("DELETE FROM usuarios")

# Inserir registros
con.sql("INSERT INTO usuarios VALUES (1, 'Lira'), (2, 'Alexandre'), (3, 'Eiko')")

print("✅ Tabela criada e dados inseridos")

In [ ]:
# Consultar os dados
print("📋 Conteúdo da tabela 'usuarios':")
con.sql("SELECT * FROM usuarios ORDER BY id").show()

# IMPORTANTE: sempre feche a conexão ao terminar
# Isso garante que o arquivo .db seja liberado corretamente
con.close()
print("\n🔒 Conexão encerrada")

> **⚠️ Boas práticas:** Sempre chame `con.close()` ao terminar. Alternativamente, use o gerenciador de contexto:
> ```python
> with duckdb.connect("meubanco.db") as con:
>     con.sql("SELECT * FROM usuarios").show()
> # A conexão é fechada automaticamente aqui
> ```

---

<a id="3.-Integração-de-Alta-Performance-com-Pandas"></a>

## 3. Integração de Alta Performance com Pandas

### 💡 Conceito

O DuckDB consegue **ler DataFrames Pandas diretamente da memória** sem cópia de dados — ele enxerga a variável Python como se fosse uma tabela SQL. Isso elimina o overhead de importação e torna as consultas extremamente rápidas.

<div style='margin:16px 0'>
<svg viewBox='0 0 620 100' xmlns='http://www.w3.org/2000/svg' style='width:100%;max-width:620px;font-family:monospace'>
  <!-- Label row -->
  <text x='90'  y='16' text-anchor='middle' fill='#8b949e' font-size='11'>DataFrame Pandas</text>
  <text x='320' y='16' text-anchor='middle' fill='#8b949e' font-size='11'>DuckDB</text>
  <text x='530' y='16' text-anchor='middle' fill='#8b949e' font-size='11'>Resultado</text>
  <!-- Box 1 -->
  <rect x='10' y='22' width='160' height='65' rx='6' fill='#161b22' stroke='#ffa657' stroke-width='1.5'/>
  <text x='90' y='50' text-anchor='middle' fill='#ffa657' font-size='13' font-weight='bold'>df_vendas</text>
  <text x='90' y='68' text-anchor='middle' fill='#8b949e' font-size='11'>(na memória)</text>
  <!-- Arrow 1 -->
  <line x1='172' y1='54' x2='218' y2='54' stroke='#58a6ff' stroke-width='1.5'/>
  <polygon points='218,50 228,54 218,58' fill='#58a6ff'/>
  <!-- Box 2 -->
  <rect x='230' y='22' width='175' height='65' rx='6' fill='#161b22' stroke='#58a6ff' stroke-width='1.5'/>
  <text x='317' y='47' text-anchor='middle' fill='#58a6ff' font-size='13' font-weight='bold'>SELECT ...</text>
  <text x='317' y='63' text-anchor='middle' fill='#8b949e' font-size='11'>FROM df_vendas</text>
  <text x='317' y='78' text-anchor='middle' fill='#8b949e' font-size='10'>WHERE / GROUP BY</text>
  <!-- Arrow 2 -->
  <line x1='407' y1='54' x2='450' y2='54' stroke='#58a6ff' stroke-width='1.5'/>
  <polygon points='450,50 460,54 450,58' fill='#58a6ff'/>
  <text x='433' y='47' text-anchor='middle' fill='#8b949e' font-size='10'>.df()</text>
  <!-- Box 3 -->
  <rect x='462' y='22' width='148' height='65' rx='6' fill='#161b22' stroke='#ffa657' stroke-width='1.5'/>
  <text x='536' y='50' text-anchor='middle' fill='#ffa657' font-size='13' font-weight='bold'>df_resultado</text>
  <text x='536' y='68' text-anchor='middle' fill='#8b949e' font-size='11'>(Pandas)</text>
</svg>
</div>


In [ ]:
# Criar um DataFrame de vendas de exemplo
df_vendas = pd.DataFrame({
    'data'     : pd.to_datetime(['2023-09-25', '2023-10-01', '2023-10-01', '2023-10-15', '2023-10-20']),
    'produto'  : ['Cadeira', 'Monitor', 'Teclado', 'Webcam', 'Mesa'],
    'categoria': ['Móveis', 'Eletrônicos', 'Acessórios', 'Eletrônicos', 'Móveis'],
    'valor'    : [770, 1479, 200, 350, 1200],
    'qtd'      : [10, 3, 7, 5, 2],
    'status'   : ['Ativo', 'Cancelado', 'Ativo', 'Ativo', 'Ativo']
})

print("📦 Dataset de vendas:")
display(df_vendas)

In [ ]:
# Executar SQL diretamente sobre a variável 'df_vendas'
# O DuckDB reconhece automaticamente o DataFrame pelo nome da variável
query = """
    SELECT 
        categoria,
        COUNT(*)          AS num_produtos,
        AVG(valor)        AS ticket_medio,
        SUM(qtd)          AS total_unidades,
        SUM(valor * qtd)  AS faturamento_total
    FROM df_vendas
    WHERE status = 'Ativo'
    GROUP BY categoria
    ORDER BY faturamento_total DESC
"""

# .df() converte o resultado de volta para um DataFrame Pandas
df_resultado = duckdb.sql(query).df()

print("📊 Análise por categoria (somente pedidos Ativos):")
display(df_resultado)

### Conversões disponíveis

| Método | Retorna |
|--------|---------|
| `.df()` | `pandas.DataFrame` |
| `.pl()` | `polars.DataFrame` (requer `pip install polars`) |
| `.arrow()` | `pyarrow.Table` |
| `.fetchall()` | `list` de tuplas Python |
| `.show()` | Exibe no terminal (sem retorno) |

---

<a id="4.-Processando-Arquivos-Grandes-com-Pouca-RAM"></a>

## 4. Processando Arquivos Grandes com Pouca RAM

### 💡 O Problema

Imagine um arquivo CSV com 50 milhões de linhas e 5 GB. O Pandas carrega **tudo na RAM** antes de filtrar:

```python
# ❌ Abordagem Pandas: ocupa 5 GB de RAM imediatamente
df = pd.read_csv("gigante.csv")                    # RAM: 5 GB
df_filtrado = df[df['status'] == 'Ativo']          # Cópia na memória
```

### ✅ A Solução com DuckDB

O DuckDB lê o arquivo em **streaming** — processa pedaço por pedaço sem carregar tudo na memória:

<div style='margin:16px 0'>
<svg viewBox='0 0 660 170' xmlns='http://www.w3.org/2000/svg' style='width:100%;max-width:660px;font-family:monospace'>
  <!-- Column labels -->
  <text x='110' y='18' text-anchor='middle' fill='#8b949e' font-size='11'>arquivo.csv (5 GB)</text>
  <text x='370' y='18' text-anchor='middle' fill='#8b949e' font-size='11'>DuckDB</text>
  <text x='570' y='18' text-anchor='middle' fill='#8b949e' font-size='11'>Resultado</text>
  <!-- CSV box -->
  <rect x='10' y='25' width='200' height='110' rx='6' fill='#161b22' stroke='#ffa657' stroke-width='1.5'/>
  <text x='110' y='55'  text-anchor='middle' fill='#c9d1d9' font-size='12'>chunk 1 (256 MB)</text>
  <line x1='20' y1='62' x2='200' y2='62' stroke='#30363d' stroke-width='1'/>
  <text x='110' y='82'  text-anchor='middle' fill='#c9d1d9' font-size='12'>chunk 2 (256 MB)</text>
  <line x1='20' y1='89' x2='200' y2='89' stroke='#30363d' stroke-width='1'/>
  <text x='110' y='109' text-anchor='middle' fill='#c9d1d9' font-size='12'>chunk 3 (256 MB)</text>
  <line x1='20' y1='116' x2='200' y2='116' stroke='#30363d' stroke-width='1'/>
  <text x='110' y='132' text-anchor='middle' fill='#8b949e' font-size='12'>...</text>
  <!-- Arrows from CSV to DuckDB -->
  <line x1='212' y1='55'  x2='278' y2='70'  stroke='#58a6ff' stroke-width='1.2'/>
  <line x1='212' y1='82'  x2='278' y2='80'  stroke='#58a6ff' stroke-width='1.2'/>
  <line x1='212' y1='109' x2='278' y2='90'  stroke='#58a6ff' stroke-width='1.2'/>
  <polygon points='278,66 272,75 282,76' fill='#58a6ff'/>
  <polygon points='278,76 272,81 282,82' fill='#58a6ff'/>
  <polygon points='278,86 272,87 282,94' fill='#58a6ff'/>
  <!-- DuckDB box -->
  <rect x='280' y='40' width='175' height='90' rx='6' fill='#161b22' stroke='#58a6ff' stroke-width='1.5'/>
  <text x='367' y='70'  text-anchor='middle' fill='#58a6ff' font-size='13' font-weight='bold'>WHERE +</text>
  <text x='367' y='88'  text-anchor='middle' fill='#58a6ff' font-size='13' font-weight='bold'>GROUP BY</text>
  <text x='367' y='116' text-anchor='middle' fill='#8b949e' font-size='10'>Máx ~256 MB RAM</text>
  <!-- Arrow DuckDB to Result -->
  <line x1='457' y1='85' x2='498' y2='85' stroke='#58a6ff' stroke-width='1.5'/>
  <polygon points='498,81 508,85 498,89' fill='#58a6ff'/>
  <text x='477' y='76' text-anchor='middle' fill='#8b949e' font-size='10'>.df()</text>
  <!-- Result box -->
  <rect x='510' y='55' width='135' height='60' rx='6' fill='#161b22' stroke='#ffa657' stroke-width='1.5'/>
  <text x='577' y='80'  text-anchor='middle' fill='#ffa657' font-size='13' font-weight='bold'>~10 linhas</text>
  <text x='577' y='98' text-anchor='middle' fill='#8b949e' font-size='11'>(kB na RAM)</text>
  <!-- Bottom labels -->
  <text x='110' y='152' text-anchor='middle' fill='#8b949e' font-size='10'>Disco (SSD)</text>
  <text x='577' y='132' text-anchor='middle' fill='#8b949e' font-size='10'>Seu DataFrame</text>
</svg>
</div>


In [ ]:
# Vamos simular um arquivo CSV grande para demonstração
import tempfile, os

NUM_LINHAS = 500_000  # Simula 500 mil registros

rng = np.random.default_rng(42)
df_simulado = pd.DataFrame({
    'produto'  : rng.choice(['Cadeira', 'Monitor', 'Teclado', 'Webcam', 'Mesa'], NUM_LINHAS),
    'categoria': rng.choice(['Móveis', 'Eletrônicos', 'Acessórios'], NUM_LINHAS),
    'valor'    : rng.integers(50, 5000, NUM_LINHAS),
    'qtd'      : rng.integers(1, 50, NUM_LINHAS),
    'status'   : rng.choice(['Ativo', 'Cancelado', 'Pendente'], NUM_LINHAS, p=[0.7, 0.2, 0.1]),
})

csv_path = "/tmp/vendas_simuladas.csv"
df_simulado.to_csv(csv_path, index=False)

tamanho_mb = os.path.getsize(csv_path) / 1024 / 1024
print(f"✅ Arquivo gerado: {csv_path}")
print(f"   📏 Linhas   : {NUM_LINHAS:,}")
print(f"   💾 Tamanho  : {tamanho_mb:.1f} MB")

In [ ]:
import time

# --- Abordagem DuckDB (streaming do CSV) ---
t0 = time.perf_counter()

df_agregado = duckdb.sql(f"""
    SELECT 
        produto,
        categoria,
        COUNT(*)          AS num_vendas,
        SUM(valor * qtd)  AS faturamento_total
    FROM '{csv_path}'
    WHERE status = 'Ativo'
    GROUP BY produto, categoria
    ORDER BY faturamento_total DESC
""").df()

tempo_duckdb = time.perf_counter() - t0

# Formatar faturamento para exibição legível (R$ 1.234.567,89)
df_exibicao = df_agregado.copy()
df_exibicao['faturamento_total'] = df_exibicao['faturamento_total'].apply(
    lambda v: f"R$ {v:_.2f}".replace('.', ',').replace('_', '.')
)

print(f"⏱️  DuckDB (streaming): {tempo_duckdb:.3f}s")
print(f"📊 Resultado ({len(df_agregado)} linhas):")
display(df_exibicao)


In [ ]:
# --- Abordagem Pandas (para comparação) ---
t0 = time.perf_counter()

df_pd = pd.read_csv(csv_path)  # Carrega tudo na RAM
df_pd_resultado = (
    df_pd[df_pd['status'] == 'Ativo']
    .assign(faturamento_total=lambda x: x['valor'] * x['qtd'])
    .groupby(['produto', 'categoria'])
    .agg(num_vendas=('valor', 'count'), faturamento_total=('faturamento_total', 'sum'))
    .reset_index()
    .sort_values('faturamento_total', ascending=False)
)

tempo_pandas = time.perf_counter() - t0

print(f"⏱️  Pandas (read_csv): {tempo_pandas:.3f}s")
print(f"\n🏆 DuckDB foi {tempo_pandas/tempo_duckdb:.1f}x mais rápido neste exemplo")

### Salvar resultado em Parquet (formato colunar eficiente)

Parquet é o formato ideal para armazenar dados processados: compressão excelente, leitura rápida por coluna, e compatível com Spark, BigQuery, Athena e afins.

In [ ]:
# Salvar o resultado processado direto em Parquet — sem passar pelo Pandas
duckdb.sql(f"""
    COPY (
        SELECT produto, categoria, SUM(valor * qtd) AS faturamento_total
        FROM '{csv_path}'
        WHERE status = 'Ativo'
        GROUP BY produto, categoria
    ) TO '/tmp/resultado.parquet' (FORMAT PARQUET)
""")

tamanho_parquet = os.path.getsize('/tmp/resultado.parquet') / 1024
print(f"✅ Salvo em '/tmp/resultado.parquet' ({tamanho_parquet:.1f} KB)")

# Ler de volta para verificar
df_check = duckdb.sql("SELECT * FROM '/tmp/resultado.parquet'").df()

# Formatar faturamento para exibição legível
df_check_exib = df_check.copy()
df_check_exib['faturamento_total'] = df_check_exib['faturamento_total'].apply(
    lambda v: f"R$ {v:_.2f}".replace('.', ',').replace('_', '.')
)
display(df_check_exib)


---

<a id="5.-Desafio-Prático"></a>

## 5. Desafio Prático

### 🎯 Enunciado

Usando o `df_vendas` criado na Seção 3, escreva **uma única query SQL** que retorne:

1. O **faturamento total** (`valor * qtd`) por produto
2. Apenas produtos com `status = 'Ativo'`
3. Apenas da categoria `'Eletrônicos'`
4. Ordenado do maior para o menor faturamento

**Resultado esperado:**

| produto | faturamento_total |
|---------|-------------------|
| Webcam  | 1750              |

---

### ✏️ Sua solução:


In [ ]:
# ✏️ Escreva sua query aqui
minha_query = """
    -- Substitua pelo seu código SQL
    SELECT 'complete o desafio!' AS mensagem
"""

df_desafio = duckdb.sql(minha_query).df()
display(df_desafio)

In [ ]:
# 💡 GABARITO — tente resolver antes de olhar!
# (Execute esta célula para ver a solução)

gabarito = """
    SELECT 
        produto,
        SUM(valor * qtd) AS faturamento_total
    FROM df_vendas
    WHERE status    = 'Ativo'
      AND categoria = 'Eletrônicos'
    GROUP BY produto
    ORDER BY faturamento_total DESC
"""

df_gab = duckdb.sql(gabarito).df()
df_gab['faturamento_total'] = df_gab['faturamento_total'].apply(
    lambda v: f"R$ {v:_.2f}".replace('.', ',').replace('_', '.')
)
display(df_gab)


---

<a id="6.-Referência-de-Comandos"></a>

## 6. Referência de Comandos

### Conexões

```python
duckdb.connect()           # Banco em memória (temporário)
duckdb.connect("file.db")  # Banco persistente em arquivo
```

### Execução de Queries

```python
duckdb.sql("SELECT ...")   # Conexão padrão (global)
con.sql("SELECT ...")      # Conexão específica
```

### Saída de Resultados

| Método | Descrição |
|--------|-----------|
| `.df()` | Converte para `pandas.DataFrame` |
| `.pl()` | Converte para `polars.DataFrame` |
| `.arrow()` | Converte para `pyarrow.Table` |
| `.fetchall()` | Retorna lista de tuplas Python |
| `.fetchone()` | Retorna a primeira linha como tupla |
| `.show()` | Exibe formatado no terminal |

### Leitura Direta de Arquivos

```python
# CSV
duckdb.sql("SELECT * FROM 'dados.csv'")

# Parquet
duckdb.sql("SELECT * FROM 'dados.parquet'")

# Múltiplos arquivos com glob
duckdb.sql("SELECT * FROM 'pasta/*.parquet'")
```

### Exportação

```python
# Parquet (recomendado)
duckdb.sql("COPY (SELECT ...) TO 'saida.parquet' (FORMAT PARQUET)")

# CSV
duckdb.sql("COPY (SELECT ...) TO 'saida.csv' (HEADER, DELIMITER ',')")
```

---

## 📚 Próximos Passos

- 📖 [Documentação oficial do DuckDB](https://duckdb.org/docs/)
- 🔌 [Extensão HTTPFS — ler arquivos direto de S3/GCS](https://duckdb.org/docs/extensions/httpfs)
- ⚡ [DuckDB + Polars — a combinação mais rápida](https://duckdb.org/docs/guides/python/polars)
- 🌐 [MotherDuck — DuckDB na nuvem](https://motherduck.com/)
